In [ ]:
library(Seurat)

In [ ]:
base_path = '/home/EOCRC_atlas/'

In [ ]:
data.path = paste0(base_path, "results/DATE_EOCRC_inferCNV/inferCNV_objects/")

In [ ]:
samples = list.dirs(path = data.path, full.names = FALSE, recursive = FALSE)
length(samples)

In [ ]:
# remove the two samples that failed inferCNV 
samples = samples[samples != "COLFR0366_T1"]
samples = samples[samples != "COLFR0034_T1"]

In [ ]:
# create data frame with cnv scores of all cells across samples
meta_colmns <- c('barcode', 'sum_abslog2_resids')
meta_allsamps <- data.frame(matrix(nrow = 0, ncol = length(meta_colmns)))
colnames(meta_allsamps) = meta_colmns

for (i in samples){
    infercnv_obj = readRDS(paste0(data.path, i, '/infercnv/run.final.infercnv_obj'))
    
    #Get magnitude of CNV summed for each cell. Take the log, then absolute value, then sum across genomic positions
    cnvs <- apply(infercnv_obj@expr.data, 2, function(x) sum(abs(log2(x))))
    
    #make dataframe with cnv scores
    df_cnv <- data.frame(barcode = names(cnvs), sum_abslog2_resids = unname(cnvs))
    
    #add cnv dataframe to meta_allsamp
    meta_allsamps <- rbind(meta_allsamps, df_cnv)
    
    print(paste0('done with ', i))
}

In [ ]:
write.table(meta_allsamps, file=paste0(base_path, "results/DATE_EOCRC_inferCNV/metadata-sum_abslog2_residuals.tsv"), quote=FALSE, sep='\t', row.names = FALSE)